# 11 · Fill interior gaps against shapely

**Question.** Does Fieldwork's `fill_interior_gaps` widget drop the right rings, and does it
report honestly how much ground the fill adds?

Dropping a polygon's interior rings is exact, so there is no tolerance to argue about. What
the check is really testing is the **bookkeeping**: the count of gaps filled, the area before
net of holes, the area after, and the difference — which is ground the traversal never
computed as reachable and the fill now claims.

The fixture is the 5 and 10 minute corridors from the Broad Street pump. Two bands rather
than one, because the example has a single pump, so one band is one polygon and a one-part
fixture could not tell *filling* from *joining up*.

Criteria fixed before running: the same gap count, no interior ring left, the filled geometry
within 0.05 m² by symmetric difference, and all three areas within 0.05 m².

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_11_fill_gaps import run

report = run(Path.cwd() / "fixtures" / "fill-gaps-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["parts"], "parts")
print(f"gaps filled     fieldwork {report['fieldwork']['gapsFilled']}   shapely {report['reference']['gapsFilled']}")
for key, label in [("areaBeforeSquareMetres", "area before"), ("areaAfterSquareMetres", "area after"),
                   ("areaAddedSquareMetres", "area added ")]:
    print(f"{label}     fieldwork {report['fieldwork'][key]:12.2f}   shapely {report['reference'][key]:12.2f}")
print()
print(f"symmetric difference of the filled geometry {report['geometryDifferenceSquareMetres']:.3e} m2")
print(f"interior rings left in the output           {report['interiorRingsRemaining']}")

## Dive deeper · what the fill claims

A gap inside a corridor is ground **no street reaches**, enclosed by streets that are
reachable: a courtyard, the inside of a block, a yard behind a wall. The traversal computed
that it cannot be reached. Filling asserts the opposite.

The widget says so in its own notes and reports the area it added. The question worth asking
of a real layer is how big that claim is, because a percentage is what a reader will act on.

In [ ]:
# How much of the filled corridor was never computed as reachable?
before = report["reference"]["areaBeforeSquareMetres"]
added = report["reference"]["areaAddedSquareMetres"]
after = report["reference"]["areaAfterSquareMetres"]
print(f"reachable corridor      {before:12.0f} m2")
print(f"gaps the fill claimed   {added:12.0f} m2")
print(f"filled polygon          {after:12.0f} m2")
print()
print(f"{added / after:.1%} of the filled polygon is ground the traversal said could not be reached.")
print(f"Anything measuring the filled layer afterwards counts it: a coverage figure taken here")
print(f"overstates reachable ground by {added / before:.1%}.")

### Per band, because the proportion is not constant

A longer band has more enclosed blocks inside it, so the fill claims more — in absolute terms
and, usually, as a share. Measuring each band separately shows whether that holds here.

In [ ]:
import numpy as np
import pyproj, shapely
from shapely.geometry import MultiPolygon, Polygon, shape

fixture = json.loads((Path.cwd() / "fixtures" / "fill-gaps-soho-001.json").read_text())
transformer = pyproj.Transformer.from_crs("OGC:CRS84", fixture["declaredParameters"]["metricCRS"], always_xy=True)

print(f"{'part':>5} {'gaps':>5} {'reachable m2':>14} {'added m2':>10} {'added share':>12}")
for index, polygon in enumerate(fixture["input"]["polygons"], start=1):
    shaped = shape(polygon["geometry"])
    parts = list(shaped.geoms) if isinstance(shaped, MultiPolygon) else [shaped]
    gaps = net = outer = 0
    for part in parts:
        projected = shapely.transform(
            part, lambda coords: np.column_stack(transformer.transform(coords[:, 0], coords[:, 1])))
        gaps += len(projected.interiors)
        net += projected.area
        outer += Polygon(projected.exterior).area
    print(f"{index:>5} {gaps:>5} {net:>14.0f} {outer - net:>10.0f} {(outer - net) / outer:>11.1%}")
print()
print("Both parts are corridors from the same pump, so the larger band contains the smaller:")
print("the rows are nested, not disjoint, and their areas must not be added together.")

### Reading the result

`agrees` means Fieldwork and shapely drop the same rings and account for the same areas. It
says nothing about whether filling is the right thing to do — that is not a question with a
numerical answer, and the reference asserts the filled ground identically, because it is the
same operation.

The widget's own guidance is the useful part: **use the unfilled polygons for coverage, and
the filled ones for a map that should read solid.**